# Step 9: Fine-tune DINOv2-Large + UperNet (Coarse 6-class)

Thin orchestration notebook for Databricks. All reusable logic lives in
`histological_image_analysis.training` (installed as a wheel on the cluster).

**Phase 1:** Coarse 6-class segmentation on 10µm Nissl CCFv3 data,
frozen DINOv2-Large backbone, UperNet head-only training.

In [ ]:
# Cell 1 — Configuration
# All environment-specific paths and hyperparameters live here.

import os

# ---------- Databricks paths ----------
WORKSPACE_BASE = "/Workspace/Users/noel.nosse@grainger.com/visual-model-ft/histology"
ONTOLOGY_PATH = f"{WORKSPACE_BASE}/ontology/1.json"
ANNOTATION_10_PATH = f"{WORKSPACE_BASE}/ccfv3/annotation_10.nrrd"
ANNOTATION_25_PATH = f"{WORKSPACE_BASE}/ccfv3/annotation_25.nrrd"
TEMPLATE_25_PATH = f"{WORKSPACE_BASE}/ccfv3/average_template_25.nrrd"
# ara_nissl_10.nrrd exceeds workspace limit — lives on DBFS
NISSL_10_PATH = "/dbfs/FileStore/allen_brain_data/ccfv3/ara_nissl_10.nrrd"

# ---------- JFrog / HuggingFace model ----------
HF_ENDPOINT = os.environ.get("HF_ENDPOINT", "https://huggingface.co")
MODEL_REPO_ID = "facebook/dinov2-large"
MODEL_CACHE_DIR = "/tmp/dinov2-large"

# ---------- Training hyperparameters ----------
NUM_LABELS = 6  # Coarse classes: Cerebrum, BS, CB, Fiber, VS, Background
CROP_SIZE = 518  # DINOv2 native resolution
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
NUM_EPOCHS = 50
FREEZE_BACKBONE = True

# ---------- Output ----------
OUTPUT_DIR = f"{WORKSPACE_BASE}/checkpoints/coarse_6class"
FINAL_MODEL_DIR = f"{WORKSPACE_BASE}/models/coarse_6class"

print(f"Ontology:    {ONTOLOGY_PATH}")
print(f"Nissl 10µm:  {NISSL_10_PATH}")
print(f"HF endpoint: {HF_ENDPOINT}")
print(f"Num labels:  {NUM_LABELS}")

In [ ]:
# Cell 2 — Download model weights from JFrog mirror
#
# Prerequisites: facebook/dinov2-large must be manually added to JFrog.
# If HF_ENDPOINT is not set, falls back to huggingface.co (works on Databricks).

from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id=MODEL_REPO_ID,
    endpoint=HF_ENDPOINT,
    token="",  # JFrog mirror requires empty token
    cache_dir=MODEL_CACHE_DIR,
)
print(f"Model downloaded to: {model_path}")

In [ ]:
# Cell 3 — Build training and validation datasets
#
# Pipeline: OntologyMapper → CCFv3Slicer → BrainSegmentationDataset

from histological_image_analysis.ontology import OntologyMapper
from histological_image_analysis.ccfv3_slicer import CCFv3Slicer
from histological_image_analysis.dataset import BrainSegmentationDataset

# 1. Load ontology and build coarse mapping
mapper = OntologyMapper(ONTOLOGY_PATH)
coarse_mapping = mapper.build_coarse_mapping()
class_names = mapper.get_class_names(coarse_mapping)
print(f"Coarse classes ({len(class_names)}): {class_names}")

# 2. Load CCFv3 volumes (10µm Nissl + annotations)
slicer = CCFv3Slicer(
    image_path=NISSL_10_PATH,
    annotation_path=ANNOTATION_10_PATH,
    ontology_mapper=mapper,
)
slicer.load_volumes()
print(f"Loaded {slicer.num_slices} coronal slices")

# 3. Spatial train/val/test split
splits = slicer.get_split_indices(train_frac=0.8, val_frac=0.1)
print(f"Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])}")

# 4. Create datasets
train_ds = BrainSegmentationDataset(
    slicer=slicer, split="train", mapping=coarse_mapping,
    crop_size=CROP_SIZE, augment=True,
)
val_ds = BrainSegmentationDataset(
    slicer=slicer, split="val", mapping=coarse_mapping,
    crop_size=CROP_SIZE, augment=False,
)
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

# Sanity check: inspect one sample
sample = train_ds[0]
print(f"pixel_values: {sample['pixel_values'].shape}, labels: {sample['labels'].shape}")

In [ ]:
# Cell 4 — Create model
#
# Phase 1: frozen backbone, head-only training.
# Forward-pass sanity check included.

import torch
from histological_image_analysis.training import create_model

model = create_model(
    num_labels=NUM_LABELS,
    freeze_backbone=FREEZE_BACKBONE,
    pretrained_backbone_path=model_path,
)

# Sanity check — single forward pass
model.eval()
with torch.no_grad():
    dummy = torch.randn(1, 3, CROP_SIZE, CROP_SIZE)
    if torch.cuda.is_available():
        model = model.cuda()
        dummy = dummy.cuda()
    out = model(pixel_values=dummy)
    print(f"Logits shape: {out.logits.shape}")
    print(f"Expected: (1, {NUM_LABELS}, {CROP_SIZE}, {CROP_SIZE})")
    assert out.logits.shape == (1, NUM_LABELS, CROP_SIZE, CROP_SIZE)
    model = model.cpu()  # Free GPU for training
print("Forward pass OK")

In [ ]:
# Cell 5 — Train

from histological_image_analysis.training import create_trainer, get_training_args

training_args = get_training_args(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    fp16=torch.cuda.is_available(),
    report_to="mlflow",
)

trainer = create_trainer(
    model=model,
    training_args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    num_labels=NUM_LABELS,
)

trainer.train()

In [ ]:
# Cell 6 — Evaluate + Visualize

import matplotlib.pyplot as plt
import numpy as np

# Run evaluation
eval_results = trainer.evaluate()
print("Evaluation results:")
for k, v in sorted(eval_results.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

# Visualize predictions vs ground truth on a few val samples
model.eval()
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for i in range(3):
    sample = val_ds[i * (len(val_ds) // 3)]
    pixel_values = sample["pixel_values"].unsqueeze(0)
    labels = sample["labels"].numpy()

    with torch.no_grad():
        if torch.cuda.is_available():
            pixel_values = pixel_values.cuda()
        logits = model(pixel_values=pixel_values).logits
        preds = logits.argmax(dim=1).squeeze().cpu().numpy()

    # Denormalize image for display
    img = sample["pixel_values"].permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    axes[i, 0].imshow(img)
    axes[i, 0].set_title("Input")
    axes[i, 1].imshow(labels, cmap="tab10", vmin=0, vmax=NUM_LABELS - 1)
    axes[i, 1].set_title("Ground Truth")
    axes[i, 2].imshow(preds, cmap="tab10", vmin=0, vmax=NUM_LABELS - 1)
    axes[i, 2].set_title("Prediction")

for ax in axes.ravel():
    ax.axis("off")

plt.suptitle(f"Coarse {NUM_LABELS}-class | mIoU={eval_results.get('eval_mean_iou', 0):.3f}")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 7 — Save final model

import os

os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
trainer.save_model(FINAL_MODEL_DIR)
print(f"Model saved to: {FINAL_MODEL_DIR}")